# COS802 Project — Housing Price Prediction
## Notebook 02: Preprocessing and Feature Engineering

This notebook converts the EDA findings into a reproducible modelling pipeline.

### Objectives
- Reuse the fixed 80/20 split from Notebook 01.
- Keep the held-out test set locked.
- Remove identifier variables (`Order`, `PID`).
- Treat structural absence in Ames Housing explicitly.
- Define original and domain-engineered feature representations.
- Define nominal and ordinal categorical preprocessing.
- Use `Pipeline` and `ColumnTransformer` so transformations are learned inside later cross-validation folds.
- Prepare raw and `log1p` target representations.
- Validate the preprocessing design without training the project regression models.

The project remains a regression study; the clustering in Notebook 01 was supplementary EDA.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")


In [ ]:
from pathlib import Path
import sys, platform
import numpy as np
import pandas as pd
import sklearn

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.model_selection import KFold


# Canonical reusable project code
PROJECT_DIR = Path("/content/drive/MyDrive/MIT BIG DATA SCIENCE/COS802_PROJECT")
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.config import SEED, TARGET, IDENTIFIERS, DATA_FILE, PROCESSED_DIR, SPLITS_DIR, FIGURES_DIR
from src.features import (
    STRUCTURAL_NONE_COLS, ENGINEERED_FEATURES, apply_structural_absence,
    AmesFeatureEngineer, validate_engineered_features, validate_feature_alignment,
)
from src.preprocessing import ORDINAL_CATEGORY_MAP, build_preprocessor
from src.evaluation import regression_metrics

SEED = 16153970
np.random.seed(SEED)

print("Python:", sys.version.split()[0])
print("Pandas:", pd.__version__)
print("Scikit-learn:", sklearn.__version__)
print("Seed:", SEED)


The `src/preprocessing.py` file has been corrected. Please re-run the previous cell (_M7jcUJxn3Yo) to import the modules without the `SyntaxError`.

In [ ]:
PROJECT_DIR = Path("/content/drive/MyDrive/MIT BIG DATA SCIENCE/COS802_PROJECT")
RAW_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DIR = PROJECT_DIR / "data" / "processed"
SPLITS_DIR = PROJECT_DIR / "data" / "splits"

DATA_FILE = RAW_DIR / "AmesHousing.txt"
SPLIT_FILE = SPLITS_DIR / "train_test_split.csv"

assert DATA_FILE.exists(), f"Missing dataset: {DATA_FILE}"
assert SPLIT_FILE.exists(), f"Missing split file: {SPLIT_FILE}"

df = pd.read_csv(DATA_FILE, sep="\t")
split_membership = pd.read_csv(SPLIT_FILE)

print("Dataset:", df.shape)
display(split_membership["split"].value_counts())


## 1. Restore the frozen split

The split created in Notebook 01 is loaded rather than regenerated. This guarantees that all subsequent experiments use exactly the same observations.


In [ ]:
TARGET = "SalePrice"
IDENTIFIERS = ["Order", "PID"]

dev_idx = split_membership.loc[
    split_membership["split"] == "development", "row_index"
].astype(int).to_numpy()

test_idx = split_membership.loc[
    split_membership["split"] == "test", "row_index"
].astype(int).to_numpy()

assert len(set(dev_idx).intersection(test_idx)) == 0
assert len(dev_idx) + len(test_idx) == len(df)

dev_df = df.loc[dev_idx].copy()
test_df = df.loc[test_idx].copy()

X_dev_raw = dev_df.drop(columns=[TARGET] + IDENTIFIERS)
y_dev_raw = dev_df[TARGET].copy()

# Test features are defined, but test target values are deliberately not inspected here.
X_test_raw = test_df.drop(columns=[TARGET] + IDENTIFIERS)

print("Development X:", X_dev_raw.shape)
print("Development y:", y_dev_raw.shape)
print("Locked test X:", X_test_raw.shape)


## 2. Missing-value semantics

Ames uses missing values for several categorical variables where `NaN` often means that the property **does not have that component**. These variables are assigned the explicit category `None` rather than being treated as unknown observations.

Remaining categorical missing values are imputed with the most frequent training-fold category. Remaining numerical missing values are imputed with the training-fold median.


In [ ]:
structural_cols_present = [c for c in STRUCTURAL_NONE_COLS if c in X_dev_raw.columns]
print("Structural-absence categorical variables:", len(structural_cols_present))
print(structural_cols_present)


In [ ]:
# Feature engineering is implemented centrally in src/features.py.
print("Using canonical AmesFeatureEngineer from src.features")


## 3. Original versus engineered representations

For the controlled RQ2 experiment:

- **Original representation:** Ames predictors after identifier removal and semantic structural-missingness handling.
- **Engineered representation:** the same predictors plus the domain-informed derived variables above.

No feature is constructed from `SalePrice`.


In [ ]:
X_dev_original = apply_structural_absence(X_dev_raw)
X_dev_engineered = AmesFeatureEngineer().fit_transform(X_dev_raw)

new_features = [c for c in X_dev_engineered.columns if c not in X_dev_original.columns]

print("Original predictor count:", X_dev_original.shape[1])
print("Engineered predictor count:", X_dev_engineered.shape[1])
print("Added engineered features:", new_features)

assert TARGET not in X_dev_engineered.columns
assert not any(c in X_dev_engineered.columns for c in IDENTIFIERS)
validate_engineered_features(X_dev_engineered)


## 4. Ordinal and nominal categorical variables

Ames contains categorical variables with genuine order (for example quality ratings). These are ordinal-encoded using explicit category orders. Other categorical variables are treated as nominal and one-hot encoded.

`OrdinalEncoder` is configured to tolerate unexpected or missing categories. The definitions below are part of the preprocessing design and will be reused in later modelling notebooks.


In [ ]:
ordinal_cols_present = [c for c in ORDINAL_CATEGORY_MAP if c in X_dev_original.columns]
print("Explicit ordinal variables:", len(ordinal_cols_present))
print(ordinal_cols_present)


In [ ]:
def feature_groups(X):
    numeric = X.select_dtypes(include=np.number).columns.tolist()
    categorical = X.select_dtypes(exclude=np.number).columns.tolist()
    ordinal = [c for c in ORDINAL_CATEGORY_MAP if c in categorical]
    nominal = [c for c in categorical if c not in ordinal]
    return {"numeric": numeric, "ordinal": ordinal, "nominal": nominal}

original_preprocessor = build_preprocessor(X_dev_original, scale_numeric=True)
engineered_preprocessor = build_preprocessor(X_dev_engineered, scale_numeric=True)
original_groups = feature_groups(X_dev_original)
engineered_groups = feature_groups(X_dev_engineered)
print("ORIGINAL:", {k: len(v) for k,v in original_groups.items()})
print("ENGINEERED:", {k: len(v) for k,v in engineered_groups.items()})


## 5. Target representations

Notebook 01 showed strong right-skew in `SalePrice` and a much more symmetric `log1p(SalePrice)` distribution. We therefore preserve both target representations for later empirical comparison.

The decision between them will be based on development-set cross-validation. Predictions from a log-target model must be transformed back with `expm1` before reporting dollar-scale MAE and RMSE.


In [ ]:
y_dev = y_dev_raw.astype(float)
y_dev_log = np.log1p(y_dev)

print(f"Raw SalePrice skewness:  {y_dev.skew():.4f}")
print(f"log1p target skewness:   {y_dev_log.skew():.4f}")

assert np.isfinite(y_dev_log).all()
assert (y_dev > 0).all()


## 6. Five-fold development cross-validation

These folds are frozen for fair model comparisons. The held-out test set is not involved. Later notebooks should use the same `cv` object or equivalent saved fold membership.


In [ ]:
cv = KFold(n_splits=5, shuffle=True, random_state=SEED)

cv_rows = []
for fold, (train_pos, val_pos) in enumerate(cv.split(X_dev_raw), start=1):
    for pos in train_pos:
        cv_rows.append({"row_index": int(dev_idx[pos]), "fold": fold, "role": "train"})
    for pos in val_pos:
        cv_rows.append({"row_index": int(dev_idx[pos]), "fold": fold, "role": "validation"})

cv_membership = pd.DataFrame(cv_rows)
cv_membership.to_csv(SPLITS_DIR / "development_cv_folds.csv", index=False)

print("Saved:", SPLITS_DIR / "development_cv_folds.csv")
display(
    cv_membership.groupby(["fold", "role"])
    .size().unstack(fill_value=0)
)


## 7. Pipeline smoke tests

We now fit the preprocessing objects **only as a technical validation** that the transformations work. No regression model is trained and no performance metric is calculated.

In later cross-validation, the preprocessor itself will be inside each model pipeline so imputation, scaling, and encoding are re-learned from each training fold.


In [ ]:
Xo = original_preprocessor.fit_transform(X_dev_original)
Xe = engineered_preprocessor.fit_transform(X_dev_engineered)

print("Original processed shape:", Xo.shape)
print("Engineered processed shape:", Xe.shape)

def nonfinite_count(X):
    if hasattr(X, "data"):  # sparse matrix
        return int((~np.isfinite(X.data)).sum())
    return int((~np.isfinite(np.asarray(X))).sum())

print("Non-finite values — original:", nonfinite_count(Xo))
print("Non-finite values — engineered:", nonfinite_count(Xe))

assert Xo.shape[0] == len(dev_df)
assert Xe.shape[0] == len(dev_df)
assert nonfinite_count(Xo) == 0
assert nonfinite_count(Xe) == 0

print("Preprocessing smoke tests passed.")


## 8. Save preprocessing design metadata

We save human-readable metadata describing the experiment. We deliberately do **not** save fitted preprocessors here, because later model pipelines must fit transformations within their own training folds.


In [ ]:
metadata_rows = []
for representation, groups in [
    ("original", original_groups),
    ("engineered", engineered_groups)
]:
    for group, cols in groups.items():
        for col in cols:
            metadata_rows.append({
                "representation": representation,
                "feature_group": group,
                "feature": col
            })

metadata = pd.DataFrame(metadata_rows)
metadata.to_csv(PROCESSED_DIR / "preprocessing_feature_groups.csv", index=False)

engineered_feature_table = pd.DataFrame({
    "engineered_feature": new_features
})
engineered_feature_table.to_csv(
    PROCESSED_DIR / "engineered_feature_list.csv", index=False
)

print("Saved preprocessing metadata.")
display(engineered_feature_table)


## Notebook 02 Conclusions

The preprocessing and feature-engineering design is now frozen for the first modelling experiments.

- The exact Notebook 01 development/test split was reused.
- `Order` and `PID` are excluded from predictors.
- Structural absence is represented explicitly for relevant Ames variables.
- Remaining numerical missingness uses median imputation.
- Remaining nominal missingness uses training-fold mode imputation followed by one-hot encoding.
- Ordered categorical variables use explicit ordinal encoding.
- Numerical predictors are standardised for models such as Ridge.
- Original and engineered representations are maintained separately for the RQ2 controlled comparison.
- Domain-engineered variables are created without using `SalePrice`.
- Raw and `log1p` target representations are available for development-only comparison.
- Five reproducible development CV folds are saved.
- No project regression model has yet been trained.
- The held-out test set remains locked.

### Next notebook

`03_baseline_models.ipynb` will establish the naive mean-price benchmark and Ridge Regression baseline, compare original versus engineered representations, and evaluate raw versus log-transformed target training using the fixed development folds.
